A generative model turns samples from a simple distribution into samples from a complex one. This section builds the smallest example that shows the idea: a one-dimensional Gaussian latent variable is pushed through a fixed function and ends up as a half-moon in the plane. The function plays the role that a neural network plays in a deep latent variable model.

The construction has three steps. The latent variable is

$$
Z \sim \mathcal{N}(0, 1).
$$

The map $\Phi$ below is the standard normal CDF. It turns the latent variable into an angle,

$$
\vartheta = \pi\,\Phi(Z) \in (0, \pi),
$$

and the generator places that angle on the unit circle,

$$
G(Z) = \big(\cos\vartheta,\ \sin\vartheta\big).
$$

Finally, isotropic Gaussian noise is added in the plane,

$$
X = G(Z) + \sigma\,\varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I_2).
$$

Because $\Phi(Z)$ is uniform on $(0, 1)$ when $Z$ is standard normal, the angle $\vartheta$ is uniform on $(0, \pi)$. The latent Gaussian is therefore transformed into an even spread of points along the upper half of the circle.

## The latent variable and the angle

The code draws the latent samples and applies the two maps. The generator, the angle map and the noise scale $\sigma$ are the only ingredients.

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)

n = 5000
sigma = 0.1


def angle(z):
    # vartheta = pi * Phi(z), with Phi the standard normal CDF
    return np.pi * stats.norm.cdf(z)


def generator(z):
    # G(z) = (cos vartheta, sin vartheta): a 1D latent mapped into the plane
    vartheta = angle(z)
    return np.column_stack((np.cos(vartheta), np.sin(vartheta)))


z = rng.standard_normal(n)
vartheta = angle(z)
curve = generator(z)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))

grid = np.linspace(-4, 4, 400)
axes[0].hist(z, bins=60, density=True, alpha=0.6)
axes[0].plot(grid, stats.norm.pdf(grid), color="black", lw=1.5)
axes[0].set_xlabel("z")
axes[0].set_ylabel("density")
axes[0].set_title("Latent Z ~ N(0, 1)")

axes[1].hist(vartheta, bins=60, range=(0, np.pi), density=True, alpha=0.6)
axes[1].axhline(1 / np.pi, color="black", lw=1.5)
axes[1].set_xlabel("angle")
axes[1].set_title("Angle: uniform on [0, pi]")

plt.tight_layout()
plt.show()

The left panel shows the latent samples with the standard normal density. The right panel shows the angles. The black line at height $1/\pi$ is the density of a uniform distribution on $[0, \pi]$.

## The curve without noise

Without the noise term, every sample is $G(Z)$ and lies exactly on the unit half-circle.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.4))
sc = ax.scatter(curve[:, 0], curve[:, 1], c=z, s=6, cmap="viridis")
fig.colorbar(sc, ax=ax, label="latent z")
ax.set_aspect("equal")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("G(Z) without noise")
plt.tight_layout()
plt.show()

The color shows the latent value that produced each point. Small $z$ lands near the right end of the arc and large $z$ near the left end. The map is smooth and one-to-one from the real line onto the open half-circle.

Here is a fact that later sections rely on. The generated distribution lives on a one-dimensional curve inside a two-dimensional space. A curve has zero area, so the probability that a sample falls in any region of positive area that avoids the curve is zero, and the distribution has no density with respect to area in the plane. You cannot write $p(\mathbf{x})$ as a function of two coordinates and integrate it over a region. A density only appears once the noise spreads the mass off the curve.

## Adding noise

Adding $\sigma\,\varepsilon$ gives each point a small Gaussian cloud around its position on the arc. The result is a half-moon whose band extends about $\pm\sigma$ around the arc.

In [ ]:
eps = rng.standard_normal((n, 2))
x = curve + sigma * eps

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.4))
ax.scatter(x[:, 0], x[:, 1], s=4, alpha=0.5)
ax.set_aspect("equal")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("X = G(Z) + sigma * eps")
plt.tight_layout()
plt.show()

With $\sigma > 0$ the distribution of $X$ has a density on the whole plane. It is the conditional Gaussian $p(\mathbf{x} \mid z) = \mathcal{N}\big(\mathbf{x};\, G(z),\, \sigma^2 I_2\big)$ averaged over the latent variable,

$$
p(\mathbf{x}) = \int \mathcal{N}\big(\mathbf{x};\, G(z),\, \sigma^2 I_2\big)\, \mathcal{N}(z;\, 0, 1)\, dz.
$$

This is the deep latent variable model of the [variational inference introduction](/aiml-common/lectures/vae/introduction/index), with a fixed function in place of a trained network. The integral has no closed form here, which is the difficulty that variational inference addresses.

The noise also matters beyond this example. Adding Gaussian noise to a distribution that lives on a curve or a surface gives it a density, and the density is smooth. Diffusion models use this fact: they add noise to data so that every intermediate distribution has a density that can be studied and learned.

## Mapping a lower dimension to a higher one

The latent space here has one dimension and the data space has two. A latent variable model can do this because $G$ is only required to be a function from the latent space to the data space. An invertible flow cannot. A flow is a smooth bijection, so its input and output have the same dimension, and the change of variables formula relates the two densities through a square Jacobian. A one-dimensional Gaussian cannot be turned into a distribution on a two-dimensional space by such a map. The latent variable model avoids the restriction by letting the noise term fill in the missing dimension.

## Angle and radius distributions

The angle $\vartheta$ is uniform on $[0, \pi]$. The Kolmogorov-Smirnov statistic, the largest gap between the empirical CDF and the uniform CDF, is of order $1/\sqrt{n}$ for $n$ samples from the uniform distribution.

The radius $R = \lVert X \rVert$ follows a Rice distribution. The generator places every point at distance 1 from the origin, so $R$ is the length of a vector with mean of length 1 plus isotropic Gaussian noise of scale $\sigma$. That length is Rice distributed with parameters $\nu = 1$ and $\sigma$. The code compares the empirical radii with this distribution.

In [ ]:
ks_angle = stats.kstest(vartheta, stats.uniform(loc=0, scale=np.pi).cdf)
print(f"angle vs uniform[0, pi]: KS = {ks_angle.statistic:.4f}, p = {ks_angle.pvalue:.3f}")
assert ks_angle.statistic < 1.63 / np.sqrt(n), "angle is not uniform on [0, pi]"

radius = np.linalg.norm(x, axis=1)
rice = stats.rice(b=1.0 / sigma, scale=sigma)
ks_radius = stats.kstest(radius, rice.cdf)
print(f"radius vs Rice(nu=1, sigma={sigma}): KS = {ks_radius.statistic:.4f}, p = {ks_radius.pvalue:.3f}")
assert ks_radius.statistic < 1.63 / np.sqrt(n), "radius does not follow the Rice distribution"

print(f"noiseless radius: min {np.linalg.norm(curve, axis=1).min():.6f}, max {np.linalg.norm(curve, axis=1).max():.6f}")
assert np.allclose(np.linalg.norm(curve, axis=1), 1.0), "G(Z) is not on the unit circle"
print("all checks passed")

The threshold $1.63/\sqrt{n}$ is the 1% critical value of the one-sample Kolmogorov-Smirnov test.